# 01 — Exploratory Data Analysis: California Housing

Goal: Understand dataset distributions, target behavior, spatial features, and identify candidate feature engineering transformations before refining the baseline model.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd


# Resolve project root
def get_project_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "configs" / "config.yaml").exists():
            return p
    return Path.cwd().resolve()


ROOT = get_project_root()
sys.path.insert(0, str(ROOT))
from src.paths import resolve

df = pd.read_csv(resolve("data/raw/california_housing.csv"))
print("Shape:", df.shape)
df.head()

## 1. Data Integrity & Missing Values
Check null counts, data types, and duplicate rows.

In [ ]:
print("Missing values:")
print(df.isna().sum())
print("\nDuplicate rows:", df.duplicated().sum())
df.info()

## 2. Target Variable Analysis (`MedHouseVal`)
Examine the distribution and summary statistics of the median house value.

In [ ]:
print(df["MedHouseVal"].describe())
print("Skewness:", df["MedHouseVal"].skew())

plt.figure(figsize=(7, 4))
plt.hist(df["MedHouseVal"], bins=50, edgecolor="black", alpha=0.7)
plt.title("Distribution of MedHouseVal ($100k)")
plt.xlabel("MedHouseVal")
plt.ylabel("Frequency")
plt.grid(True, alpha=0.3)
plt.show()

## 3. Correlations with Target
Which features correlate most strongly with median house value?

In [ ]:
corr = df.corr()["MedHouseVal"].sort_values(ascending=False)
print("Correlations with MedHouseVal:")
print(corr)

plt.figure(figsize=(7, 4))
plt.scatter(df["MedInc"], df["MedHouseVal"], alpha=0.2, s=5)
plt.title("Median Income vs Median House Value")
plt.xlabel("MedInc ($10k)")
plt.ylabel("MedHouseVal ($100k)")
plt.grid(True, alpha=0.3)
plt.show()

## 4. Spatial Analysis (Latitude & Longitude)
Plot house values across California geographic coordinates.

In [ ]:
plt.figure(figsize=(8, 6))
scatter = plt.scatter(
    df["Longitude"],
    df["Latitude"],
    c=df["MedHouseVal"],
    cmap="viridis",
    s=df["Population"] / 100,
    alpha=0.4,
)
plt.colorbar(scatter, label="MedHouseVal ($100k)")
plt.title("California Housing Prices (Color = Price, Size = Population)")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.grid(True, alpha=0.3)
plt.show()

## 5. Key Findings & Engineering Roadmap
1. **Target Capping**: The ceiling at 5.0 ($500,000) causes an artificial upper spike in target distribution.
2. **Income Dominance**: `MedInc` has the highest linear correlation (~0.69) with housing values.
3. **Spatial Clusters**: Strong price clustering near Bay Area and LA coastline.
4. **Candidate Features for Iteration 2**:
   - `rooms_per_household = AveRooms / AveOccup`
   - `bedrooms_ratio = AveBedrms / AveRooms`
   - Log transform skewed features like `Population`.